# `ngsplot`: NGSolve's `Draw`, as a matplotlib picture

`ngsolve.webgui.Draw` draws a finite element function beautifully, and then puts it inside a
browser widget you cannot script, screenshot from a script, or hand to a file.  `ngsplot.Plot`
paints the same picture -- the same colour ramp and band quantisation, the same Phong shading,
the same thick lines with their 4x MSAA, the same camera -- into an ordinary matplotlib figure,
and hands back the `AxesImage`.

Run this notebook from the first cell to the last.  What it needs:

```sh
python -m pip install -e .        # ngsplot, numpy and matplotlib
python -m pip install ngsolve     # to build the meshes there is something to draw
```

The first cell also puts `src/` of the checkout on the path, so the notebook runs from a checkout
that was never installed.  There is no widget in sight: every figure below is a figure you can
resize, retitle, put next to another one and `savefig`.

In [ ]:
import importlib.util
import sys
from pathlib import Path

if importlib.util.find_spec("ngsplot") is None:  # a checkout that was never installed
    for candidate in (Path.cwd() / "src", Path.cwd().parent / "src"):
        if (candidate / "ngsplot").is_dir():
            sys.path.insert(0, str(candidate))
            break

%matplotlib inline

import matplotlib
import netgen.libngpy._meshing as ngmesh
import numpy as np
import ngsolve as ngs
import ngsplot
from matplotlib import pyplot as mpl
from ngsplot import Plot, pyplot

print(f"ngsolve {ngs.__version__}, ngsplot {ngsplot.__version__}, "
      f"matplotlib {matplotlib.__version__}")

## 1D: a line, and a function on it

`netgen` gives no `unit_line` to Python, so the mesh is made from nodes and elements.  A dim-1
mesh is a mesh like any other, and `H1` on it behaves as it does anywhere else.

In [ ]:
def line_mesh(n=40):
    """n equal segments of the unit interval, as a mesh of dimension 1."""
    mesh = ngmesh.Mesh(1)
    mesh.AddPoints(np.array([[i / n, 0.0, 0.0] for i in range(n + 1)]))
    mesh.AddElements(1, 1, np.array([[i + 1, i + 2] for i in range(n)], dtype=np.int32), base=1)
    return ngs.Mesh(mesh)


mesh1 = line_mesh(40)
u1 = ngs.GridFunction(ngs.H1(mesh1, order=3))
u1.Set(ngs.sin(2 * np.pi * ngs.x) * ngs.exp(-2 * ngs.x))

Plot(u1)
pyplot.gcf()

The curve comes out flat and black, which is what the page does with it: on a 1D mesh the
renderer neither lifts the geometry nor colours it, so `line_width`, `edges` and `wireframe` are
the only knobs that move any ink, and `vmin`/`vmax` have nothing to say.  (Colouring a 1D curve
along its values is the one place where `ngsplot` will eventually differ from the page on
purpose; `docs/SPEC.md` and the README both record it as such.)

In [ ]:
figure, axes = mpl.subplots(1, 2, figsize=(11, 3.8))

Plot(u1, ax=axes[0])
axes[0].set_title("as the page draws it")

Plot(u1, ax=axes[1], line_width=1, edges=False, wireframe=False)
axes[1].set_title("line_width=1, no mesh lines")

figure.subplots_adjust(wspace=0.05)
figure

## 2D: the surface case, which is what `Draw` is mostly used for

The generators are lowercase (`ngs.unit_square`, `ngs.unit_cube`), and the picture is what
`Draw(gfu)` would have put on the screen -- with a colourbar that is a real matplotlib colourbar,
in the same bands and between the same ends the panel's two number fields hold.

In [ ]:
mesh2 = ngs.Mesh(ngs.unit_square.GenerateMesh(maxh=0.25))
u2 = ngs.GridFunction(ngs.H1(mesh2, order=3))
u2.Set(ngs.sin(np.pi * ngs.x) * ngs.sin(np.pi * ngs.y))

Plot(u2, colorbar=True)
pyplot.gcf()

### The knobs

Everything the panel has a control for is a keyword, and the rule for a keyword is one line
long: the ones the renderer takes (`vmin`, `vmax`, `ncolors`, `cmap`, `wireframe`, `edges`,
`line_width`, `euler_angles`, `deformation`, `dark_backside`, `function_mode`, ...) go to the
painter, and the rest (`order`, `draw_vol`, `draw_surf`, `settings`, `scale`, `nodal_p1`) go to
`Draw`, which shapes the payload with them.  `eval` is the one exception: it names which part of
a value to colour (`"x"`, `"norm"`, `"real"`, `"imag"`), and it is what `Draw` *cannot* be asked
for -- see the section "Why Draw is clunky" in `docs/SPEC.md`.

In [ ]:
figure, axes = mpl.subplots(2, 2, figsize=(11.4, 7.4))

first = Plot(u2, ax=axes[0, 0], vmin=0, vmax=1, ncolors=8)
pyplot.colorbar(first)
axes[0, 0].set_title("vmin, vmax, ncolors=8, with a bar")

Plot(u2, ax=axes[0, 1], cmap="viridis", wireframe=False)
axes[0, 1].set_title('cmap="viridis", wireframe=False')

Plot(u2, ax=axes[1, 0], line_width=1, edges=False)
axes[1, 0].set_title("line_width=1, edges=False")

Plot(u2, ax=axes[1, 1], euler_angles=(35, 25, 0), deformation=0.12)
axes[1, 1].set_title("euler_angles=(35, 25, 0), deformation=0.12")

figure.subplots_adjust(wspace=0.25, hspace=0.25)
figure

### What else arrives at the same picture

A `CoefficientFunction` belongs to no mesh, so it is drawn on the one you hand it -- and hears
from ngsolve, in words, when you forget to.  A `Mesh` with nothing on it is `Plot(mesh)`, or
`Plot(mesh=mesh)`, which is `Draw(mesh)`: the green of the viewer, its own shading, no scale and
so no colourbar to ask for.  A vector field is coloured by its norm unless `eval` says otherwise,
a complex one by its real part, and both are the same call.

In [ ]:
figure, axes = mpl.subplots(1, 3, figsize=(13.5, 4.4))

Plot(ngs.exp(-((ngs.x - 0.5) ** 2 + (ngs.y - 0.3) ** 2) * 30), ax=axes[0], mesh=mesh2)
axes[0].set_title("a CoefficientFunction, on the mesh it is given")

gfv = ngs.GridFunction(ngs.H1(mesh2, order=2, dim=2))
gfv.Set((ngs.y, -ngs.x))
Plot(gfv, ax=axes[1], eval="x")
axes[1].set_title('a vector field, eval="x"')

Plot(ax=axes[2], mesh=mesh2)
axes[2].set_title("the mesh on its own")

figure.subplots_adjust(wspace=0.05)
figure

In [ ]:
figure, axes = mpl.subplots(1, 3, figsize=(13.5, 4.4))

gfc = ngs.GridFunction(ngs.H1(mesh2, order=2, complex=True))
gfc.Set(ngs.x + 1j * ngs.y)
Plot(gfc, ax=axes[0], eval="imag")
axes[0].set_title('a complex field, eval="imag"')

fes = ngs.L2(mesh2, order=0)  # one value per element, the piecewise-constant case
gfl = ngs.GridFunction(fes)
for i, ed in enumerate(mesh2.Elements()):
    for dof in fes.GetDofNrs(ngs.ElementId(ed)):
        gfl.vec[dof] = i
Plot(gfl, ax=axes[1], ncolors=16)
axes[1].set_title("L2(order=0), 16 bands")

Plot(ngs.GridFunction(ngs.NumberSpace(mesh2)), ax=axes[2], vmin=0, vmax=3)
axes[2].set_title("NumberSpace: one value for the whole mesh")

figure.subplots_adjust(wspace=0.05)
figure

## 3D: a solid

The same call: a `GridFunction` on a solid mesh arrives with `draw_vol` set by `Draw`, so the
picture is the shaded surface of the solid, seen from wherever the camera is pointed.
`dark_backside` is the knob that makes a closed solid read as closed, and `cmap="ngsolve"` is the
page's own ramp, which matplotlib has never heard of and `ngsplot` cuts to the band count asked
for without registering anything anywhere.

In [ ]:
mesh3 = ngs.Mesh(ngs.unit_cube.GenerateMesh(maxh=0.5))
u3 = ngs.GridFunction(ngs.H1(mesh3, order=2))
u3.Set(ngs.x + 2 * ngs.y + 3 * ngs.z)

Plot(u3, colorbar=True)
pyplot.gcf()

In [ ]:
figure, axes = mpl.subplots(1, 3, figsize=(13.5, 4.4))

Plot(u3, ax=axes[0], euler_angles=(30, 20, 0), dark_backside=True)
axes[0].set_title("the dragged view, dark_backside=True")

Plot(u3, ax=axes[1], euler_angles=(35, 45, 0), cmap="ngsolve", ncolors=16, wireframe=False)
axes[1].set_title('cmap="ngsolve", 16 bands, no wireframe')

Plot(u3, ax=axes[2], cmap="viridis", vmin=1, vmax=5, edges=False)
axes[2].set_title("viridis over a range of its own")

figure.subplots_adjust(wspace=0.05)
figure

## A time loop into frames, and frames into a movie

Plotting twice into one axis replaces the picture instead of laying a second one under it, and
`savefig` of the current figure comes out at the figure's own dpi -- byte for byte the array the
renderer made.  That is the whole of the animation story: write frames, then let ffmpeg have
them.

In [ ]:
frames = Path("frames")
frames.mkdir(exist_ok=True)

mesh_t = ngs.Mesh(ngs.unit_square.GenerateMesh(maxh=0.2))
ut = ngs.GridFunction(ngs.H1(mesh_t, order=2))
for step in range(12):
    t = step / 11
    ut.Set((1 - 0.6 * t) * ngs.sin(np.pi * (ngs.x - 0.25 * t)) * ngs.sin(np.pi * ngs.y))
    Plot(ut, vmin=-1, vmax=1)  # a fixed range, so that the frames are comparable
    pyplot.savefig(frames / f"frame_{step:02d}.png")

print(sorted(p.name for p in frames.glob("frame_*.png"))[:4], "...")

pyplot.close("all")  # the inline backend shows every figure still open at the end of a cell
figure, axes = mpl.subplots(1, 4, figsize=(13.5, 3.6))
for k, step in enumerate((0, 4, 8, 11)):
    axes[k].imshow(mpl.imread(frames / f"frame_{step:02d}.png"))
    axes[k].set_title(f"t = {step / 11:.2f}")
    axes[k].axis("off")
figure.subplots_adjust(wspace=0.02)
figure

```sh
ffmpeg -framerate 12 -i frames/frame_%02d.png -pix_fmt yuv420p ngsplot.mp4
```

## From here

`help(Plot)` is the whole surface, and `from ngsplot import Plot` is the only import a script
needs; `ngsplot.pyplot` holds the rest of it (`figure`, `gcf`, `gca`, `sca`, `cla`, `close`,
`show`, `savefig`, `colorbar`).  `docs/SPEC.md` records what every pixel is doing and where in
`webgui@0.2.39` it was read from, including the two deliberate deviations from the page: no GUI
panel, and a 1D function drawn as flatly as the page draws it.